# Reference cases, similarity, and evidence priors

**MedLIME Learning Track — Notebook 10/12** · Curriculum weeks 29–32 · Suggested time: 60–100 minutes

**Before starting:** Notebook 04; array distances and averaging.

**Learning goal:** Build a reference-only evidence library and use weighted neighbor explanations as a prior.

> **Important:** All images below are synthetic visual teaching phantoms, **not real medical images**. The educational implementations are **not** the original MedLIME implementation, are not evaluated for clinical use, and do not establish diagnostic or explanation reliability.


In [ ]:
from pathlib import Path
import sys
here = Path.cwd()
for candidate in (here, here.parent, here.parent.parent):
    if (candidate / 'medlime_tutorials.py').exists():
        sys.path.insert(0, str(candidate.resolve())); break
else:
    raise FileNotFoundError('Open Jupyter from the MedLIME folder or its notebooks/ subfolder')
import numpy as np
import matplotlib.pyplot as plt
from medlime_tutorials import *
np.set_printoptions(precision=3, suppress=True)


## How might past images help?
Evidence-based regularization aims to stabilize explanations by using related reference examples. Here we use cosine similarity on pooled intensities, retrieve top-k library images, calculate their LIME maps, and average the patch coefficients.

Crucially, **reference images must be separate from held-out queries**. This lesson uses no query labels, query lesion masks, or nearest-neighbor ground truth to compute the prior. It is still a simplified approximation of paper EBR.

In [ ]:
library=np.stack([synthetic_image(1000+i,True)[0] for i in range(18)])
query,ground_truth=synthetic_image(seed=71,lesion=True)
idx,sims=nearest_reference(query,library,k=4)
print('Selected reference indices:',idx,'similarities:',np.round(sims,4))
show_images([query,*library[idx]],['held-out query']+[f'neighbor {j}' for j in idx])
plt.show()

## Each neighbor has an explanation
Compute local coefficients for the reference images using the same black-box predictor, perturbation budget, and mask strategy.

In [ ]:
reference_coef=[]
for ref in library:
    fit=lime_explain(ref,n=75,seed=19)
    reference_coef.append(fit['coef'])
reference_coef=np.asarray(reference_coef)
print('Reference explanations array:',reference_coef.shape)
prior,ids,weights=evidence_prior(query,library,reference_coef,k=4,bandwidth=.05)
print('Neighbor contribution weights:',np.round(weights,3))
show_images([coef_map(reference_coef[j]) for j in ids]+[coef_map(prior)],
 [f'neighbor {j}' for j in ids]+['evidence prior'],cmap='viridis')
plt.show()

## A statistical and ethical caution
A similar *looking* image may have different pathology. Similarity in a pooled intensity space is not clinical equivalence, and using a reference pool with mislabeled or duplicate patients can invalidate the analysis.

In [ ]:
print('Prior coefficient spread:',round(float(prior.min()),4),'to',round(float(prior.max()),4))
for bandwidth in [.01,.05,.2]:
    p,_,w=evidence_prior(query,library,reference_coef,k=4,bandwidth=bandwidth)
    print('bandwidth',bandwidth,'effective max neighbor weight',round(float(w.max()),3),
          'prior norm',round(float(np.linalg.norm(p)),3))

## What you should now understand
- A similarity metric maps each image to a neighborhood of reference examples.
- Neighbor explanations can form a soft prior, not absolute ground truth.
- Query/reference separation and provenance matter for valid evaluation.

## Try it yourself (do not just read the code)
1. Change the number of neighbors from 2 to 10; plot the evidence prior.
2. Insert a duplicated query into the reference library; describe why this would be a leakage risk.

## Exit ticket
**Question:** How could systematic differences between scanners make intensity-based nearest neighbors misleading?

Record your answers in a new Markdown cell before moving to the next notebook. Many questions have more than one reasonable answer; explain your evidence.